# D1.4 · Detection engineering *for* agents

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.3 · Agent-assisted detection engineering](https://spbreed.github.io/cyber-commons/lessons/D1.3.html)**.

| | |
|---|---|
| Tools used | Falco, Sigma |

## What this lesson is

**What it covers.** Write five detections for agent misbehaviour and fire each one.

**Why a security engineer needs it.** Scope drift, unusual tool sequencing, off-hours autonomous action. The control it builds is: detections whose subject is a non-human principal.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Writing a detection for an agent means writing one where machine-speed behaviour is normal and the baseline has no human rhythm in it at all. Every heuristic that relies on tiredness, working hours or typing speed is gone.

> **At CyberTravels.** Writing a detection where the subject is CyberTravels means writing one where 1,400 actions an hour is normal and every heuristic that relies on human rhythm is gone.

## 2 · The framework

```
   human baseline                agent baseline
   +-------------------+         +----------------------+
   | works 9-6         |         | works always         |
   | 12 actions/hour   |         | 1400 actions/hour    |
   | makes typos       |         | never retries a typo |
   +-------------------+         +----------------------+

   detect on SHAPE, not volume: new resource classes, new tool
   sequences, a spike in distinct destinations
```

This is the new work, and it starts by discarding baselines that have served the
SOC well for twenty years.

Human behavioural detection assumes irregularity, working hours, and a rate
ceiling set by typing speed. An agent violates all three *while behaving
correctly*:

| Classic signal | For a human | For an agent |
|---|---|---|
| two countries in an hour | incident | routine (multi-region) |
| 300 file reads a minute | incident | idle |
| activity at 03:00 | suspicious | meaningless |
| the same action 500 times | suspicious | a stuck loop — but not malicious |

Applying human baselines to agents produces an alert on every session, so the
rule gets tuned down, and then it never fires again — including when something
is genuinely wrong.

The signals that *do* work for agents are about **change**: a tool it has never
used, a mix that has shifted, a scope exercised that was never needed before.

## 3 · Verify — the alert text an analyst can act on

"Anomaly detected" fails both tests: it does not say what changed, and it does not say what to do.

## 4 · The procedure, as a skill

All three classic rules fire on CyberTravels' patch agent doing exactly its job, and only the rate rule fires on the human. The skill runs both, then measures drift from a signed-off baseline week by week — naming the new tool rather than reporting a distance.

### The skill — [`skills/detection/agent-aware-rule-review/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-aware-rule-review/SKILL.md)

```yaml
name: agent-aware-rule-review
description: >-
  Run existing detection rules against an agent doing its job to find which fire
  on legitimate work, and measure behavioural drift from a signed-off baseline
  week by week. Use when agents enter an estate whose detection content predates
  them.
allowed-tools: Read, Grep, Glob
```

# Every classic rule fires on an agent working normally

Detection content written for humans classifies agent behaviour as an incident:
the volume rule, the off-hours rule, the breadth-of-access rule. Turning them
off loses coverage; leaving them on trains the analyst to ignore them. The
answer is a per-population baseline and a drift measure against it.

## When to use this

Whenever agents are introduced into an estate with existing detection content,
and at each model or manifest upgrade afterwards.

## Procedure

**1 — Run each classic rule against a clean agent run** and against a human's
day. Record which fire on which. Rules that fire on the agent and not the human
are the ones needing a population.

**2 — Sign off a baseline for each agent.** Tools used, resources touched, rate,
hours. Signed off means somebody agreed it — a baseline nobody approved is a
measurement, not a control.

**3 — Compare each subsequent week against the baseline.** Report a drift figure
and, more usefully, the *new* items: a tool that was not in the baseline is the
signal, and it is legible in a way a distance number is not.

**4 — Set a tolerance and say what crossing it does.** Drift within tolerance is
noise; beyond it is a review, and the review is of the change that caused it —
usually a model upgrade or a manifest edit.

**5 — Write the alert text for a human.** "patch-agent drift 0.35, new tools:
run_shell" is actionable. A distance metric on its own gets closed.

## Output contract

```json
{
  "classic_rules": [{"name": "str", "fires_on_agent": true, "fires_on_human": false}],
  "baseline": {"agent": "str", "tools": ["str"], "resources": ["str"], "signed_off": true},
  "weeks": [{"week": 0, "drift": 0.0, "new_tools": ["str"], "within_tolerance": true}],
  "tolerance": 0.0,
  "alert_text": "str"
}
```

## Failure modes

- **Muting the classic rules.** You lose them for humans too.
- **A baseline nobody signed off.** Nothing to appeal to when it drifts.
- **Alerting on the distance only.** Name the new tool.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-aware-rule-review/scripts/agent_aware_rule_review.py
SCRIPT = "skills/detection/agent-aware-rule-review/scripts/agent_aware_rule_review.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

All three classic rules fire on an agent doing its job and only the rate rule fires on the human. Drift is within tolerance at week 1, significant at week 4 with `write_file` and `repo:write` new, and larger at week 8 with `run_shell` and an `exec` scope. The alert text names what changed, why it matters and what to do.

## Your turn

Take one human-baseline rule in your SIEM and check how it behaves against a service account. If it fires nightly, it is already tuned off for that actor — which means you have no detection there at all.

---

**Next → [D1.5 · Agent telemetry as a data source](https://spbreed.github.io/cyber-commons/lessons/D1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*